In [ ]:
import warnings
import numpy as np
import scipy.sparse as sp
from scipy.sparse import kron as sp_kron
import cupy as cp
import cupyx.scipy.sparse as cp_sp
import time
start_time = time.perf_counter()

warnings.filterwarnings("ignore", category=sp.SparseEfficiencyWarning)



class NVCenterSystemBaseGPU:

    def __init__(self, params):
        self.N = params['N']
        self.D = params['D']
        self.omega_0 = params['omega_0']
        self.gamma_e = params['gamma_e']
        self.Bz = params['Bz']
        self.zeta = params['zeta']
        self.g = params['g']

        # Single-spin operators constructed on CPU to avoid GPU VRAM exhaustion during expansion
        self.I3 = sp.eye(3, dtype=complex, format='csr')
        self.Sz = sp.csr_matrix(
            np.array([[1, 0, 0], [0, 0, 0], [0, 0, -1]], dtype=complex)
        )
        self.Sx = sp.csr_matrix(
            (1 / np.sqrt(2))
            * np.array([[0, 1, 0], [1, 0, 1], [0, 1, 0]], dtype=complex)
        )
        self.Sy_prime = sp.csr_matrix(
            (1 / np.sqrt(2))
            * np.array([[0, -1j, 0], [1j, 0, -1j], [0, 1j, 0]], dtype=complex)
        )

    def _expand_operator_sparse(self, op, target_index):
        """Expands a 3x3 operator onto the N-spin Hilbert space in CPU system memory."""
        result = op if target_index == 0 else self.I3
        for i in range(1, self.N):
            current_op = op if i == target_index else self.I3
            result = sp_kron(result, current_op, format='csr')
        return result

    def get_control_hamiltonian(self):
        dim = 3**self.N
        H_control = sp.csr_matrix((dim, dim), dtype=complex)
        cos_z = np.cos(self.zeta)
        sin_z = np.sin(self.zeta)

        for j in range(self.N):
            Sx_j = self._expand_operator_sparse(self.Sx, j)
            Sy_p_j = self._expand_operator_sparse(self.Sy_prime, j)
            H_control = H_control + 0.5 * (cos_z * Sx_j + sin_z * Sy_p_j)

        return H_control

    def get_ground_state(self):
        ms_0 = np.array([0, 1, 0], dtype=complex)
        ground_state = ms_0
        for _ in range(1, self.N):
            ground_state = np.kron(ground_state, ms_0)
        return ground_state

    def _euler_solve_optimized(self, H_drift, H_control, u_array, y0, t_total, num_steps):
        #Executes this integration on GPU.
        dt = t_total / num_steps
        c = -1j  #2nd prob: remove 2pi to match how dynamiqs uses the schrond eqn
        psi = y0.copy()

        for i in range(num_steps):
            u_t = u_array[i]
            k1 = c * (H_drift.dot(psi) + u_t * H_control.dot(psi))
            psi += dt * k1
        #psi = psi / cp.linalg.norm(psi)    #1st prob:solves the explosion that happened but cant add it bec by doing so we're just manipulating the code

        return psi

    def evolve_state_optimized(self, initial_state, t_total, control_pulse, t_array=None, time_steps=2000):
        H_drift_cpu = self.get_drift_hamiltonian()
        H_control_cpu = self.get_control_hamiltonian()

        H_drift = cp_sp.csr_matrix(H_drift_cpu)
        H_control = cp_sp.csr_matrix(H_control_cpu)

        initial_state_gpu = cp.asarray(initial_state).reshape(-1, 1)

        # Generate timestamps for Euler (1 per step)
        euler_time_points = cp.linspace(0, t_total, time_steps, endpoint=False)

        if callable(control_pulse):
            t_np = cp.asnumpy(euler_time_points)
            u_array = cp.asarray(control_pulse(t_np))
        elif isinstance(control_pulse, (np.ndarray, cp.ndarray)):
            cp_control = cp.asarray(control_pulse)
            cp_t_array = cp.asarray(t_array)
            u_array = cp.interp(euler_time_points, cp_t_array, cp_control)
        else:
            raise TypeError("control_pulse must be a callable or array.")

        return self._euler_solve_optimized(
            H_drift, H_control, u_array, initial_state_gpu, t_total, num_steps=time_steps
        )

class NVCenterNearestNeighborGPU(NVCenterSystemBaseGPU):

    def get_drift_hamiltonian(self):
        dim = 3**self.N
        H_drift = sp.csr_matrix((dim, dim), dtype=complex)

        Sz_sq = self.Sz.dot(self.Sz)

        for j in range(self.N):
            Sz_j_sq = self._expand_operator_sparse(Sz_sq, j)
            Sz_j = self._expand_operator_sparse(self.Sz, j)

            term1 = (self.D[j] - self.omega_0) * Sz_j_sq
            term2 = self.gamma_e * self.Bz[j] * Sz_j
            H_drift = H_drift + (term1 - term2)

        for j in range(self.N - 1):
            Sz_j = self._expand_operator_sparse(self.Sz, j)
            Sz_j_next = self._expand_operator_sparse(self.Sz, j + 1)
            H_drift = H_drift + self.g[j] * Sz_j.dot(Sz_j_next)

        return H_drift


# ==========================================
# Execution Block
# ==========================================
if __name__ == "__main__":
    N_test = 7
    test_params = {
        'N': N_test,
        'D': [2870.0] * N_test,
        'omega_0': 2870.0,
        'gamma_e': 28000.0,
        'Bz': [1e-3] * N_test,
        'zeta': np.pi / 4,
        'g': [0.1] * (N_test - 1),
    }

    print("Building Sparse Hamiltonians on CPU...")
    nv_chain = NVCenterNearestNeighborGPU(test_params)
    initial_state_cpu = nv_chain.get_ground_state()
    t_pulse = 1.0

    print("\n--- Testing Lambda Function Pulse ---")
    lambda_pulse = lambda t: 2.0 * np.sin(np.pi * (t / t_pulse))

    final_state_lambda = nv_chain.evolve_state_optimized(
        initial_state=initial_state_cpu,
        t_total=t_pulse,
        control_pulse=lambda_pulse,
        time_steps=2000,
    )

    norm_lambda = cp.asnumpy(cp.linalg.norm(final_state_lambda))
    print("Norm check (Lambda):", norm_lambda)

    print("\n--- Testing Array Pulse ---")
    time_points = np.linspace(0, t_pulse, 100)
    array_pulse = 2.0 * np.sin(np.pi * (time_points / t_pulse))

    final_state_array = nv_chain.evolve_state_optimized(
        initial_state=initial_state_cpu,
        t_total=t_pulse,
        control_pulse=array_pulse,
        t_array=time_points,
        time_steps=2000,
    )

    norm_array = cp.asnumpy(cp.linalg.norm(final_state_array))
    print("Norm check (Array):", norm_array)
    end_time = time.perf_counter()
print(f"Time taken: {end_time - start_time:.4f} seconds")



Building Sparse Hamiltonians on CPU...

--- Testing Lambda Function Pulse ---
Norm check (Lambda): 1.0000396521787602

--- Testing Array Pulse ---
Norm check (Array): 1.0000396455444862
Time taken: 2.6892 seconds
